In [276]:
from collections import Counter
from datasets import load_dataset
from sklearn.metrics import classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from nltk import word_tokenize
from nltk.corpus import cmudict
from nltk.util import bigrams, trigrams




Load and preprocess data from huggingface:

In [358]:
train_data = load_dataset("derek-thomas/ScienceQA", split= 'train')
train_data = train_data.remove_columns(['image','choices','hint','answer','task','subject','topic','category','skill'])

dev_data = load_dataset("derek-thomas/ScienceQA", split= 'validation')
dev_data = dev_data.remove_columns(['image','choices','hint','answer','task','subject','topic','category','skill'])

test_data = load_dataset("derek-thomas/ScienceQA", split= 'test')
test_data = test_data.remove_columns(['image','choices','hint','answer','task','subject','topic','category','skill'])


BUILD FEATURES:

In [291]:
training_unigrams = []
training_bigrams = []
training_trigrams = []
training_labels = []

for instance in train_data:
    tokens = word_tokenize(instance['lecture'])
    tokens += word_tokenize(instance['question'])
    bigram_list = list(bigrams(tokens))
    trigram_list = list(trigrams(tokens))
    unigram_feature_dict = {x:float(0) for x in tokens}
    bigram_feature_dict = {x:float(0) for x in bigram_list}
    trigram_feature_dict = {x:float(0) for x in trigram_list}
    training_labels.append(instance['grade'])
    
    for tok in tokens:
        unigram_feature_dict[tok]+= 1.0
    training_unigrams.append(unigram_feature_dict)
    for bigram in bigram_list:
        bigram_feature_dict[bigram] += 1.0
    training_bigrams.append(bigram_feature_dict)
    for trigram in trigram_list:
        trigram_feature_dict[trigram] += 1.0
    training_trigrams.append(trigram_feature_dict)

In [292]:
dev_unigrams = []
dev_bigrams = []
dev_trigrams = []
dev_labels = []

for instance in dev_data:
    tokens = word_tokenize(instance['lecture'])
    tokens += word_tokenize(instance['question'])
    bigram_list = list(bigrams(tokens))
    trigram_list = list(trigrams(tokens))
    unigram_feature_dict = {x:float(0) for x in tokens}
    bigram_feature_dict = {x:float(0) for x in bigram_list}
    trigram_feature_dict = {x:float(0) for x in trigram_list}
    dev_labels.append(instance['grade'])
    
    for tok in tokens:
        unigram_feature_dict[tok]+= 1.0
    dev_unigrams.append(unigram_feature_dict)
    for bigram in bigram_list:
        bigram_feature_dict[bigram] += 1.0
    dev_bigrams.append(bigram_feature_dict)
    for trigram in trigram_list:
        trigram_feature_dict[trigram] += 1.0
    dev_trigrams.append(trigram_feature_dict)
    


In [293]:
test_unigrams = []
test_bigrams = []
test_trigrams = []
test_labels = []

for instance in test_data:
    tokens = word_tokenize(instance['lecture'])
    tokens += word_tokenize(instance['question'])
    bigram_list = list(bigrams(tokens))
    trigram_list = list(trigrams(tokens))
    unigram_feature_dict = {x:float(0) for x in tokens}
    bigram_feature_dict = {x:float(0) for x in bigram_list}
    trigram_feature_dict = {x:float(0) for x in trigram_list}
    test_labels.append(instance['grade'])
    
    for tok in tokens:
        unigram_feature_dict[tok]+= 1.0
    test_unigrams.append(unigram_feature_dict)
    for bigram in bigram_list:
        bigram_feature_dict[bigram] += 1.0
    test_bigrams.append(bigram_feature_dict)
    for trigram in trigram_list:
        trigram_feature_dict[trigram] += 1.0
    test_trigrams.append(trigram_feature_dict)

Training and Testing:

Unigrams:

In [345]:
vectorizer = DictVectorizer()
classifier = MultinomialNB()
mnb_alpha_values = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
score_dict = {}

X_train_vector = vectorizer.fit_transform(training_unigrams)
y_train = training_labels
classifier.fit(X_train_vector,y_train)




print(classifier.score(X_train_vector,y_train))

0.48892032060348894


In [87]:
X_dev_vector = vectorizer.transform(dev_unigrams)
y_dev = dev_labels
score_dict = {}
for i in mnb_alpha_values:
    classifier = MultinomialNB(alpha = i)
    classifier.fit(X_dev_vector, y_dev)
    score_dict[i] = classifier.score(X_dev_vector,y_dev)


for i in score_dict:
    print(i, score_dict[i])



0.1 0.5253477953312898
0.2 0.5232256543268097
0.3 0.5163876444234851
0.4 0.5112001886347559
0.5 0.5097854279651026
0.6 0.5076632869606225
0.7 0.5088422541853337
0.8 0.5062485262909691
0.9 0.5027116246168356
1.0 0.5005894836123556


In [317]:
classifier = MultinomialNB(alpha = 0.1)
X_test_vector = vectorizer.transform(test_unigrams)
y_test = test_labels
classifier.fit(X_test_vector, y_test)
print(classification_report(y_test,classifier.predict(X_test_vector)))

              precision    recall  f1-score   support

      grade1       0.57      1.00      0.72        17
     grade10       0.38      0.63      0.47       119
     grade11       0.73      0.58      0.65       105
     grade12       0.68      0.55      0.61        86
      grade2       0.46      0.86      0.60       293
      grade3       0.64      0.55      0.60       638
      grade4       0.60      0.40      0.48       673
      grade5       0.49      0.72      0.58       644
      grade6       0.52      0.52      0.52       459
      grade7       0.63      0.41      0.50       566
      grade8       0.72      0.36      0.48       519
      grade9       0.37      0.88      0.52       122

    accuracy                           0.54      4241
   macro avg       0.57      0.62      0.56      4241
weighted avg       0.58      0.54      0.54      4241



Bigrams:

In [346]:
X_train_vector = vectorizer.fit_transform(training_bigrams)
y_train = training_labels
classifier.fit(X_train_vector,y_train)

classifier.score(X_train_vector,y_train) 

0.5411755461260411

In [91]:
X_dev_vector = vectorizer.transform(dev_bigrams)
y_dev = dev_labels
score_dict = {}
for i in mnb_alpha_values:
    classifier = MultinomialNB(alpha = i)
    classifier.fit(X_dev_vector, y_dev)
    score_dict[i] = classifier.score(X_dev_vector,y_dev)


for i in score_dict:
    print(i, score_dict[i])



0.1 0.5713275170950247
0.2 0.5628389530771044
0.3 0.5595378448479132
0.4 0.5567083235086064
0.5 0.5543503890591841
0.6 0.5578872907333177
0.7 0.556236736618722
0.8 0.5576514972883754
0.9 0.5564725300636643
1.0 0.556236736618722


In [319]:
X_test_vector = vectorizer.transform(test_bigrams)
y_test = test_labels

classifier = MultinomialNB(alpha=0.1)
classifier.fit(X_test_vector,y_test)
print(classification_report(y_test,classifier.predict(X_test_vector)))

              precision    recall  f1-score   support

      grade1       0.55      0.94      0.70        17
     grade10       0.38      0.63      0.47       119
     grade11       0.71      0.60      0.65       105
     grade12       0.66      0.53      0.59        86
      grade2       0.64      0.88      0.74       293
      grade3       0.56      0.63      0.59       638
      grade4       0.64      0.43      0.51       673
      grade5       0.52      0.70      0.60       644
      grade6       0.50      0.59      0.54       459
      grade7       0.62      0.42      0.50       566
      grade8       0.67      0.29      0.41       519
      grade9       0.40      0.89      0.55       122

    accuracy                           0.56      4241
   macro avg       0.57      0.63      0.57      4241
weighted avg       0.58      0.56      0.55      4241



Trigrams:

In [321]:
X_train_vector = vectorizer.fit_transform(training_trigrams)
y_train = training_labels
classifier.fit(X_train_vector,y_train)

classifier.score(X_train_vector, training_labels)

0.5721357850070722

In [364]:
X_dev_vector = vectorizer.transform(dev_trigrams)
y_dev = dev_data['grade']
score_dict = {}
for i in mnb_alpha_values:
    classifier = MultinomialNB(alpha = i)
    classifier.fit(X_dev_vector, y_dev)
    score_dict[i] = classifier.score(X_dev_vector,y_dev)


for i in score_dict:
    print(i, score_dict[i])



0.1 0.17377976892242394
0.2 0.17377976892242394
0.3 0.17377976892242394
0.4 0.17377976892242394
0.5 0.17377976892242394
0.6 0.17377976892242394
0.7 0.17377976892242394
0.8 0.17377976892242394
0.9 0.17377976892242394
1.0 0.17377976892242394


In [323]:
X_test_vector = vectorizer.transform(test_trigrams)
y_test = test_labels

classifier = MultinomialNB(alpha=0.1)
classifier.fit(X_test_vector,y_test)
print(classification_report(y_test,classifier.predict(X_test_vector)))

              precision    recall  f1-score   support

      grade1       0.80      0.94      0.86        17
     grade10       0.39      0.63      0.48       119
     grade11       0.68      0.56      0.61       105
     grade12       0.66      0.55      0.60        86
      grade2       0.65      0.92      0.76       293
      grade3       0.58      0.67      0.62       638
      grade4       0.63      0.42      0.51       673
      grade5       0.54      0.70      0.61       644
      grade6       0.48      0.59      0.53       459
      grade7       0.61      0.41      0.49       566
      grade8       0.64      0.27      0.38       519
      grade9       0.41      0.89      0.56       122

    accuracy                           0.56      4241
   macro avg       0.59      0.63      0.58      4241
weighted avg       0.58      0.56      0.55      4241



Phonemes/Syllables:


In [354]:
phoneme_dict = cmudict.dict()
digits = ['0','1','2','3','4','5','6','7','8','9']

In [369]:
train_phonemes = []

for instance in train_data:
    features = {}
    instance_token_count = 0
    instance_type_set = set()
    instance_phoneme_count = 0
    
    tokens = word_tokenize(instance['lecture']) + word_tokenize(instance['question'])
    for token in tokens:
        instance_token_count += 1
        instance_type_set.add(token)
        if token in phoneme_dict:
            instance_phoneme_count += len(phoneme_dict[token][0])
    
    features['ttr'] = len(instance_type_set)/instance_token_count
    features['phonemes_per_word'] = instance_phoneme_count/instance_token_count
    
    train_phonemes.append(features)
            

    

In [371]:
dev_phonemes = []
dev_labels = []
for instance in dev_data:
    features = {}
    instance_token_count = 0
    instance_type_set = set()
    instance_phoneme_count = 0
    
    syllable_count = 0
    dev_labels += instance['grade']
    tokens = word_tokenize(instance['lecture']) + word_tokenize(instance['question'])
    for token in tokens:
        instance_token_count += 1
        instance_type_set.add(token)

        if token.lower() in phoneme_dict:
            instance_phoneme_count += len(phoneme_dict[token.lower()][0])
            for phoneme in phoneme_dict[token.lower()][0]:
                if phoneme[-1] in digits:
                    syllable_count += 1
            else:
                if token.lower() not in phoneme_dict:
                    syllable_count += 1
                
    
    features['ttr'] = len(instance_type_set)/instance_token_count
    features['phonemes_per_word'] = instance_phoneme_count/instance_token_count
    features['syllables_per_word'] = syllable_count/instance_token_count
    
    dev_phonemes.append(features)
            


In [350]:
test_phonemes = []
test_labels = []
for instance in test_data:
    features = {}
    instance_token_count = 0
    instance_type_set = set()
    instance_phoneme_count = 0
    instance_character_count = 0
    syllable_count = 0
    test_labels += instance['grade']
    tokens = word_tokenize(instance['lecture']) + word_tokenize(instance['question'])
    for token in tokens:
        instance_token_count += 1
        instance_type_set.add(token)
        for char in token:
            instance_character_count += 1
        if token.lower() in phoneme_dict:
            instance_phoneme_count += len(phoneme_dict[token.lower()][0])
            for phoneme in phoneme_dict[token.lower()][0]:
                if phoneme[-1] in digits:
                    syllable_count += 1
    
    features['ttr'] = len(instance_type_set)/instance_token_count
    features['phonemes_per_word'] = instance_phoneme_count/instance_token_count
    features['syllables_per_word'] = syllable_count/instance_token_count
    
    test_phonemes.append(features)

In [370]:
classifier = MultinomialNB()
vectorizer = DictVectorizer()
X_train_phoneme_vector = vectorizer.fit_transform(train_phonemes,train_data['grade'])

classifier.fit(X_train_phoneme_vector,y_train)

classifier.score(X_train_phoneme_vector,y_train)

0.1676881973911677

In [372]:
phoneme_score_dict = {}
for i in mnb_alpha_values:
    classifier = MultinomialNB(alpha= i)
    X_dev_phonemes_vector = vectorizer.transform(dev_phonemes)
    classifier.fit(X_dev_phonemes_vector,dev_data['grade'])
    phoneme_score_dict[i] = classifier.score(X_dev_phonemes_vector,dev_data['grade'])

for i in phoneme_score_dict:
    print(i, phoneme_score_dict[i])

    

0.1 0.17377976892242394
0.2 0.17377976892242394
0.3 0.17377976892242394
0.4 0.17377976892242394
0.5 0.17377976892242394
0.6 0.17377976892242394
0.7 0.17377976892242394
0.8 0.17377976892242394
0.9 0.17377976892242394
1.0 0.17377976892242394


{'ttr': 0.7684210526315789, 'phonemes_per_word': 3.526315789473684, 'syllables_per_word': 1.4105263157894736}


In [367]:
X_test_phonemes_vector = vectorizer.transform(test_phonemes)
y_test_phonemes = test_data['grade']

classifier = MultinomialNB(alpha=0.1)
classifier.fit(X_test_phonemes_vector, y_test)
print(classification_report(y_test_phonemes,classifier.predict(X_test_phonemes_vector)))

              precision    recall  f1-score   support

      grade1       0.00      0.00      0.00        17
     grade10       0.00      0.00      0.00       119
     grade11       0.00      0.00      0.00       105
     grade12       0.00      0.00      0.00        86
      grade2       0.00      0.00      0.00       293
      grade3       0.00      0.00      0.00       638
      grade4       0.16      1.00      0.27       673
      grade5       0.00      0.00      0.00       644
      grade6       0.00      0.00      0.00       459
      grade7       0.00      0.00      0.00       566
      grade8       0.00      0.00      0.00       519
      grade9       0.00      0.00      0.00       122

    accuracy                           0.16      4241
   macro avg       0.01      0.08      0.02      4241
weighted avg       0.03      0.16      0.04      4241



C:\Users\capta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\capta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\capta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classificati

Logistic Regression:

LR - Unigrams:

In [383]:
classifier = LogisticRegression(max_iter=12500)
X_train_vector = vectorizer.fit_transform(training_unigrams)
y_train = train_data['grade']
classifier.fit(X_train_vector,y_train)

print(classifier.score(X_train_vector,y_train))


0.7398239823982399


In [384]:
X_dev_vector = vectorizer.fit_transform(dev_unigrams)
y_dev = dev_data['grade']
classifier.fit(X_dev_vector,y_dev)

c_param = [1, 5, 10, 25, 50, 100]
solvers = ['lbfgs','newton-cg','liblinear','sag','saga']
lr_score_dict = {}

for solver in solvers:
    lr_score_dict[solver] = {x:0 for x in c_param}
    for c in c_param:
        classifier = LogisticRegression(solver= solver, C=c, max_iter=12500)
        classifier.fit(X_dev_vector,y_dev)
        lr_score_dict[solver][c] = classifier.score(X_dev_vector,y_dev)

    



In [389]:
X_test_vector = vectorizer.transform(test_unigrams)
y_test = test_data['grade']
classifier = LogisticRegression(solver= 'lbfgs', C=25, max_iter=12500)
classifier.fit(X_test_vector,y_test)

print(classification_report(y_test,classifier.predict(X_test_vector)))


              precision    recall  f1-score   support

      grade1       0.85      1.00      0.92        17
     grade10       0.74      0.91      0.82       119
     grade11       0.87      0.83      0.85       105
     grade12       1.00      0.83      0.90        86
      grade2       0.91      0.86      0.88       293
      grade3       0.78      0.82      0.80       638
      grade4       0.84      0.75      0.79       673
      grade5       0.73      0.81      0.77       644
      grade6       0.72      0.76      0.74       459
      grade7       0.76      0.72      0.74       566
      grade8       0.77      0.73      0.75       519
      grade9       0.86      0.94      0.90       122

    accuracy                           0.79      4241
   macro avg       0.82      0.83      0.82      4241
weighted avg       0.79      0.79      0.79      4241



LR - Bigrams

0.7803708942322803


In [391]:
X_dev_vector = vectorizer.fit_transform(dev_bigrams)
y_dev = dev_data['grade']
classifier.fit(X_dev_vector,y_dev)

c_param = [1, 5, 10, 25, 50, 100]
solvers = ['lbfgs','newton-cg','liblinear','sag','saga']
lr_score_dict = {}

for solver in solvers:
    lr_score_dict[solver] = {x:0 for x in c_param}
    for c in c_param:
        classifier = LogisticRegression(solver= solver, C=c, max_iter=12500)
        classifier.fit(X_dev_vector,y_dev)
        lr_score_dict[solver][c] = classifier.score(X_dev_vector,y_dev)

    

print(lr_score_dict)

{'lbfgs': {1: 0.8120726243810422, 5: 0.8139589719405801, 10: 0.8139589719405801, 25: 0.8137231784956378, 50: 0.8139589719405801, 100: 0.8134873850506956}, 'newton-cg': {1: 0.8120726243810422, 5: 0.8141947653855223, 10: 0.8141947653855223, 25: 0.8134873850506956, 50: 0.8134873850506956, 100: 0.8134873850506956}, 'liblinear': {1: 0.8090073095967932, 5: 0.8144305588304646, 10: 0.8141947653855223, 25: 0.8137231784956378, 50: 0.8137231784956378, 100: 0.8134873850506956}, 'sag': {1: 0.8118368309360999, 5: 0.8141947653855223, 10: 0.8144305588304646, 25: 0.8144305588304646, 50: 0.8141947653855223, 100: 0.8139589719405801}, 'saga': {1: 0.8108936571563311, 5: 0.8141947653855223, 10: 0.8144305588304646, 25: 0.8144305588304646, 50: 0.8144305588304646, 100: 0.8144305588304646}}


In [392]:
X_test_vector = vectorizer.transform(test_bigrams)
y_test = test_data['grade']
classifier = LogisticRegression(solver= 'liblinear', C=5, max_iter=12500)
classifier.fit(X_test_vector,y_test)

print(classification_report(y_test,classifier.predict(X_test_vector)))


              precision    recall  f1-score   support

      grade1       1.00      0.82      0.90        17
     grade10       0.67      0.88      0.76       119
     grade11       0.80      0.77      0.79       105
     grade12       0.95      0.70      0.81        86
      grade2       0.89      0.86      0.88       293
      grade3       0.80      0.77      0.78       638
      grade4       0.81      0.71      0.75       673
      grade5       0.69      0.76      0.72       644
      grade6       0.58      0.72      0.64       459
      grade7       0.75      0.60      0.67       566
      grade8       0.61      0.66      0.63       519
      grade9       0.80      0.86      0.83       122

    accuracy                           0.73      4241
   macro avg       0.78      0.76      0.76      4241
weighted avg       0.74      0.73      0.73      4241



LR - Trigrams


In [393]:
classifier = LogisticRegression(max_iter=12500)
X_train_vector = vectorizer.fit_transform(training_trigrams)
y_train = train_data['grade']
classifier.fit(X_train_vector,y_train)

print(classifier.score(X_train_vector,y_train))


0.7826496935407826


In [394]:
X_dev_vector = vectorizer.fit_transform(dev_trigrams)
y_dev = dev_data['grade']
classifier.fit(X_dev_vector,y_dev)

c_param = [1, 5, 10, 25, 50, 100]
solvers = ['lbfgs','newton-cg','liblinear','sag','saga']
lr_score_dict = {}

for solver in solvers:
    lr_score_dict[solver] = {x:0 for x in c_param}
    for c in c_param:
        classifier = LogisticRegression(solver= solver, C=c, max_iter=12500)
        classifier.fit(X_dev_vector,y_dev)
        lr_score_dict[solver][c] = classifier.score(X_dev_vector,y_dev)

    

print(lr_score_dict)

{'lbfgs': {1: 0.8130157981608112, 5: 0.8141947653855223, 10: 0.8137231784956378, 25: 0.8141947653855223, 50: 0.8134873850506956, 100: 0.8139589719405801}, 'newton-cg': {1: 0.8130157981608112, 5: 0.8141947653855223, 10: 0.8139589719405801, 25: 0.8134873850506956, 50: 0.8134873850506956, 100: 0.8134873850506956}, 'liblinear': {1: 0.8113652440462155, 5: 0.8141947653855223, 10: 0.8141947653855223, 25: 0.8139589719405801, 50: 0.8137231784956378, 100: 0.8134873850506956}, 'sag': {1: 0.8130157981608112, 5: 0.8141947653855223, 10: 0.8141947653855223, 25: 0.8144305588304646, 50: 0.8139589719405801, 100: 0.8137231784956378}, 'saga': {1: 0.8130157981608112, 5: 0.8141947653855223, 10: 0.8144305588304646, 25: 0.8144305588304646, 50: 0.8144305588304646, 100: 0.8144305588304646}}


In [396]:
X_test_vector = vectorizer.transform(test_trigrams)
y_test = test_data['grade']
classifier = LogisticRegression(solver= 'saga', C=10, max_iter=12500)
classifier.fit(X_test_vector,y_test)

print(classification_report(y_test,classifier.predict(X_test_vector)))

              precision    recall  f1-score   support

      grade1       1.00      0.76      0.87        17
     grade10       0.64      0.85      0.73       119
     grade11       0.76      0.70      0.73       105
     grade12       0.89      0.63      0.73        86
      grade2       0.85      0.83      0.84       293
      grade3       0.73      0.76      0.75       638
      grade4       0.78      0.68      0.73       673
      grade5       0.67      0.74      0.70       644
      grade6       0.61      0.63      0.62       459
      grade7       0.72      0.57      0.64       566
      grade8       0.56      0.65      0.60       519
      grade9       0.73      0.80      0.77       122

    accuracy                           0.70      4241
   macro avg       0.74      0.72      0.73      4241
weighted avg       0.70      0.70      0.70      4241



LR - Syllables

In [397]:
X_train_phonemes_vector = vectorizer.transform(train_phonemes)
y_train = train_data['grade']
classifier.fit(X_train_phonemes_vector,y_train)

print(classifier.score(X_train_phonemes_vector,y_train))

0.14034260568914034


In [399]:
X_dev_syllables_vector = vectorizer.fit_transform(dev_phonemes)
y_dev = dev_data['grade']
classifier.fit(X_dev_syllables_vector,y_dev)

c_param = [1, 5, 10, 25, 50, 100]
solvers = ['lbfgs','newton-cg','liblinear','sag','saga']
lr_score_dict = {}

for solver in solvers:
    lr_score_dict[solver] = {x:0 for x in c_param}
    for c in c_param:
        classifier = LogisticRegression(solver= solver, C=c, max_iter=12500)
        classifier.fit(X_dev_syllables_vector,y_dev)
        lr_score_dict[solver][c] = classifier.score(X_dev_syllables_vector,y_dev)

    

print(lr_score_dict)

{'lbfgs': {1: 0.21150672011318086, 5: 0.20962037255364302, 10: 0.20844140532893185, 25: 0.20773402499410518, 50: 0.20891299221881632, 100: 0.20844140532893185}, 'newton-cg': {1: 0.212685687337892, 5: 0.20985616599858523, 10: 0.20867719877387408, 25: 0.20867719877387408, 50: 0.20796981843904738, 100: 0.20796981843904738}, 'liblinear': {1: 0.21999528413110114, 5: 0.21457203489742985, 10: 0.21221410044800754, 25: 0.212685687337892, 50: 0.212685687337892, 100: 0.21244989389294977}, 'sag': {1: 0.21221410044800754, 5: 0.21009195944352746, 10: 0.20914878566375855, 25: 0.20914878566375855, 50: 0.20938457910870079, 100: 0.20914878566375855}, 'saga': {1: 0.21386465456260315, 5: 0.2133930676727187, 10: 0.2110351332232964, 25: 0.21244989389294977, 50: 0.2119783070030653, 100: 0.21221410044800754}}


In [401]:
X_test_syllables_vector = vectorizer.transform(test_phonemes)
y_test = test_data['grade']
classifier = LogisticRegression(solver= 'liblinear', C=1, max_iter=12500)
classifier.fit(X_test_syllables_vector,y_test)

print(classification_report(y_test,classifier.predict(X_test_syllables_vector)))

              precision    recall  f1-score   support

      grade1       0.00      0.00      0.00        17
     grade10       0.00      0.00      0.00       119
     grade11       0.00      0.00      0.00       105
     grade12       0.00      0.00      0.00        86
      grade2       0.27      0.13      0.17       293
      grade3       0.13      0.09      0.10       638
      grade4       0.19      0.34      0.25       673
      grade5       0.21      0.26      0.23       644
      grade6       0.00      0.00      0.00       459
      grade7       0.21      0.57      0.31       566
      grade8       0.24      0.07      0.11       519
      grade9       0.00      0.00      0.00       122

    accuracy                           0.20      4241
   macro avg       0.10      0.12      0.10      4241
weighted avg       0.16      0.20      0.16      4241



C:\Users\capta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\capta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\capta\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classificati